In [1]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

# Use the kagglehub client library to attach Kaggle resources like competitions, datasets, and models to your session
# Learn more about kagglehub: https://github.com/Kaggle/kagglehub/blob/main/README.md

import kagglehub
# kagglehub.dataset_download('<owner>/<dataset-slug>')

/kaggle/input/competitions/otto-recommender-system/sample_submission.csv
/kaggle/input/competitions/otto-recommender-system/test.jsonl
/kaggle/input/competitions/otto-recommender-system/train.jsonl


In [2]:
import json
from collections import Counter
import pandas as pd

TRAIN = "/kaggle/input/competitions/otto-recommender-system/train.jsonl"
TEST  = "/kaggle/input/competitions/otto-recommender-system/test.jsonl"
MAX_SESSIONS = 1_000_000      # on ne lit pas tout, pour aller vite

counter = Counter()
with open(TRAIN) as f:
    for i, line in enumerate(f):
        if i >= MAX_SESSIONS:
            break
        for e in json.loads(line)["events"]:
            counter[e["aid"]] += 1
top20 = " ".join(str(a) for a, _ in counter.most_common(20))

rows = []
with open(TEST) as f:
    for line in f:
        sid = json.loads(line)["session"]
        for t in ("clicks", "carts", "orders"):
            rows.append({"session_type": f"{sid}_{t}", "labels": top20})

pd.DataFrame(rows).to_csv("submission.csv", index=False)

In [3]:
import os
print(os.path.exists("/kaggle"))                 # False = means we're not in Kaggle
for root, dirs, files in os.walk("/kaggle/input"):
    print(root, files[:5])
    if root.count("/") >= 4: dirs.clear()        # do not descend too deeply

True
/kaggle/input []
/kaggle/input/competitions []
/kaggle/input/competitions/otto-recommender-system ['sample_submission.csv', 'test.jsonl', 'train.jsonl']


In [4]:
del counter
import gc; gc.collect()

0

**Matrice de co-visitation (le plus gros gain)**
L’idée : si deux articles apparaissent souvent dans les mêmes sessions, on recommande l’un quand la session contient l’autre. Les articles récents de la session pèsent plus, et on complète avec les articles populaires si besoin.

In [5]:
import json
from collections import defaultdict
import pandas as pd

TRAIN = "/kaggle/input/competitions/otto-recommender-system/train.jsonl"
TEST  = "/kaggle/input/competitions/otto-recommender-system/test.jsonl"
MAX_SESSIONS = 600_000      # augmentez progressivement si la mémoire suit
CHUNK, LAST_N, TOP_PER_ITEM = 20_000, 20, 20

def read_sessions(path, max_sessions=None):
    sess, aids, tss = [], [], []
    with open(path) as f:
        for i, line in enumerate(f):
            if max_sessions and i >= max_sessions: break
            s = json.loads(line)
            for e in s["events"]:
                sess.append(s["session"]); aids.append(e["aid"]); tss.append(e["ts"])
    return pd.DataFrame({"session": sess, "aid": aids, "ts": tss})

train = read_sessions(TRAIN, MAX_SESSIONS)
popular = train["aid"].value_counts().index[:20].tolist()

# matrice de co-visitation, calculée par blocs de sessions (pour économiser la mémoire)
total = None
sids = train["session"].unique()
for k in range(0, len(sids), CHUNK):
    d = train[train["session"].isin(sids[k:k+CHUNK])]
    d = d.sort_values(["session", "ts"]).groupby("session").tail(LAST_N)
    d = d.drop_duplicates(["session", "aid"])[["session", "aid"]]
    pairs = d.merge(d, on="session")
    pairs = pairs[pairs["aid_x"] != pairs["aid_y"]]
    cnt = pairs.groupby(["aid_x", "aid_y"]).size()
    total = cnt if total is None else total.add(cnt, fill_value=0)

total = total.reset_index(name="w").sort_values(["aid_x", "w"], ascending=[True, False])
total = total.groupby("aid_x").head(TOP_PER_ITEM)
covisit = {a: list(zip(g["aid_y"], g["w"])) for a, g in total.groupby("aid_x")}

SEEN_N = 10 # essayez 0, 3, 5, 10 et comparez les scores

def recommend(aids):
    seen = list(dict.fromkeys(reversed(aids)))[:LAST_N]
    scores = defaultdict(float)
    for rank, a in enumerate(seen):
        for b, w in covisit.get(a, []):
            scores[b] += w / (1 + rank)
    first = seen[:SEEN_N]                                    # les derniers articles de la session
    best = [b for b, _ in sorted(scores.items(), key=lambda x: -x[1]) if b not in first]
    final = (first + best)[:20]
    for p in popular:                                        # compléter à 20
        if len(final) >= 20: break
        if p not in final: final.append(p)
    return final[:20]
    
rows = []
with open(TEST) as f:
    for line in f:
        s = json.loads(line)
        labels = " ".join(str(int(a)) for a in recommend([e["aid"] for e in s["events"]]))
        for t in ("clicks", "carts", "orders"):
            rows.append({"session_type": f"{s['session']}_{t}", "labels": labels})

pd.DataFrame(rows).to_csv("submission.csv", index=False)

In [6]:
from gensim.models import Word2Vec     # si ImportError : !pip install gensim
import numpy as np

MIN_COUNT, W2V_DIM, TOPN_W2V = 5, 64, 20

sessions = (train.sort_values(["session", "ts"]).groupby("session")["aid"]
            .apply(lambda s: [str(a) for a in s]).tolist())

w2v = Word2Vec(sessions, vector_size=W2V_DIM, window=5, min_count=MIN_COUNT,
               sg=1, workers=4, epochs=3, seed=0)
del sessions

kv = w2v.wv
vecs = kv.vectors / np.linalg.norm(kv.vectors, axis=1, keepdims=True)
keys = np.array([int(k) for k in kv.index_to_key])

# pour chaque article, ses 20 voisins les p"lus proches (par blocs pour économiser la mémoire)
w2v_nb = {}
k = min(TOPN_W2V, len(keys) - 1)
for s in range(0, len(vecs), 1000):
    sims = vecs[s:s+1000] @ vecs.T
    top = np.argpartition(-sims, k, axis=1)[:, :k+1]
    for i in range(sims.shape[0]):
        idx = top[i][np.argsort(-sims[i, top[i]])]
        idx = [j for j in idx if j != s + i][:k]
        w2v_nb[int(keys[s + i])] = [(int(keys[j]), float(sims[i, j])) for j in idx]
print(len(w2v_nb), "articles avec voisins")

628457 articles avec voisins


In [7]:
SEEN_N = 10      # j'ai met la meilleur valeur qui donne lE meilleur score

def recommend(aids):
    seen = list(dict.fromkeys(reversed(aids)))[:LAST_N]
    scores = defaultdict(float)
    for rank, a in enumerate(seen):
        for b, w in covisit.get(a, []):
            scores[b] += w / (1 + rank)
    first = seen[:SEEN_N]
    best = [b for b, _ in sorted(scores.items(), key=lambda x: -x[1]) if b not in first]
    final = (first + best)[:20]

    if len(final) < 20:                                  # compléter avec les voisins word2vec
        extra = defaultdict(float)
        for rank, a in enumerate(seen):
            for b, s in w2v_nb.get(a, []):
                extra[b] += s / (1 + rank)
        for b, _ in sorted(extra.items(), key=lambda x: -x[1]):
            if len(final) >= 20: break
            if b not in final: final.append(b)

    for p in popular:                                    # en dernier recours : populaires
        if len(final) >= 20: break
        if p not in final: final.append(p)
    return [int(x) for x in final[:20]]

rows = []
with open(TEST) as f:
    for line in f:
        s = json.loads(line)
        labels = " ".join(str(a) for a in recommend([e["aid"] for e in s["events"]]))
        for t in ("clicks", "carts", "orders"):
            rows.append({"session_type": f"{s['session']}_{t}", "labels": labels})

pd.DataFrame(rows).to_csv("submission.csv", index=False)